In [ ]:
!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=fb99204a00167713142bc145d90a79ddd3f5beb3e508da5e22373e5a7229d889
  Stored in directory: /root/.cache/pip/wheels/36/5d/63/8243a7ee78fff0f944d638fd0e66d7278888f5e2285d7346b6
Successfully built odfpy


In [ ]:
import pandas as pd
import json
import uuid
import requests
import os
import time
import re
import zipfile

# --- CONFIGURACIÓN ---
API_KEY = "20462364-4650be092c1702f82e17ad513"
ODS_FILE_PATH = '/content/Inventario CEP Las Palmas Completo.ods' # Asegúrate que el nombre de tu archivo sea correcto
# ---------------------

def clean_activity_name(name):
    """Elimina patrones no deseados del nombre de la actividad."""
    if not isinstance(name, str):
        return ""
    cleaned_name = re.sub(r"\(\+\d+\)", "", name).strip()
    return cleaned_name

def search_pixabay_illustration(query):
    """Busca una ilustración en Pixabay y devuelve la URL."""
    base_url = "https://pixabay.com/api/"
    params = {
        "key": API_KEY, "q": query, "image_type": "illustration",
        "per_page": 3, "safesearch": "true", "lang": "es"
    }
    try:
        response = requests.get(base_url, params=params)
        if response.status_code == 429:
            print("Límite de API alcanzado. Esperando 60s...")
            time.sleep(60)
            response = requests.get(base_url, params=params)

        if response.status_code == 200:
            data = response.json()
            if data.get("hits"):
                return data["hits"][0]["webformatURL"]
    except Exception as e:
        print(f"Error en la petición a Pixabay: {e}")
    return None

def download_image(url, save_path):
    """Descarga una imagen desde una URL."""
    try:
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
    except Exception as e:
        print(f"Error descargando imagen {url}: {e}")

# --- Preparación del entorno ---
os.makedirs('images', exist_ok=True)
try:
    df = pd.read_excel(ODS_FILE_PATH, engine='odf')
    df.columns = df.columns.str.strip() # Limpiar espacios en nombres de columnas
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo en {ODS_FILE_PATH}")
    exit()

# --- PLANTILLA H5P ADAPTADA CON EL CAMPO "TÍTULO" ---
h5p_template = {
    "infoWall": {
        "propertiesGroup": {
            "properties": [
                # Las propiedades deben coincidir en número y orden con la lista "entries" de cada panel
                {"label": "Título", "showLabel": False, "searchInProperty": True, "styling": {"bold": True, "italic": False}},
                {"label": "Descripción", "showLabel": False, "searchInProperty": True, "styling": {"bold": False, "italic": False}},
                {"label": "Proyecto", "showLabel": True, "searchInProperty": True, "styling": {"bold": True, "italic": False}},
                {"label": "Categoría", "showLabel": True, "searchInProperty": True, "styling": {"bold": True, "italic": False}},
                {"label": "Etiquetas", "showLabel": True, "searchInProperty": True, "styling": {"bold": False, "italic": True}}
            ]
        },
        "panels": []
    },
    "behaviour": {
        "useFallbackImage": False, "imageWidth": 150, "imageHeight": 150,
        "alternateBackground": True, "offerFilterField": True, "modeFilterField": "and"
    },
    "l10n": {
        "noEntriesError": "The author did not enter anything.",
        "noMatchesForFilter": "There are not matches for @query.",
        "enterToFilter": "Enter a query to filter the content for relevant entries.",
        "listChanged": "List changed. Showing @visible of @total items."
    },
    "header": "Catálogo de Actividades"
}

print("Procesando actividades desde el archivo ODS...")

for index, row in df.iterrows():
    # 1. Obtener y validar el Nombre (para título del panel y búsqueda de imagen)
    original_name = row.get("Nombre")
    if pd.isna(original_name) or not str(original_name).strip():
        print(f"Fila {index+2}: Nombre vacío. Saltando.")
        continue

    print(f"Procesando: {original_name}")
    cleaned_name = clean_activity_name(str(original_name))

    # 2. Gestión de imagen
    illustration_url = search_pixabay_illustration(cleaned_name) or "https://via.placeholder.com/400x267?text=Sin+Imagen"
    image_filename = f"{re.sub(r'[^a-zA-Z0-9]', '_', cleaned_name)[:20]}_{uuid.uuid4().hex[:6]}.jpg"
    image_path = os.path.join('images', image_filename)
    download_image(illustration_url, image_path)

    # 3. Preparar los datos de texto de cada campo
    # Si 'Título' está vacío, usamos 'Nombre' como fallback
    titulo_txt = str(row.get("Título")) if pd.notna(row.get("Título")) and str(row.get("Título")).strip() else str(original_name)

    descripcion_txt = str(row.get("Descripción", "")) if pd.notna(row.get("Descripción")) else ""
    proyecto_txt = str(row.get("Proyecto", "")) if pd.notna(row.get("Proyecto")) else ""
    categoria_txt = str(row.get("Categoría", "")) if pd.notna(row.get("Categoría")) else ""
    etiquetas_txt = str(row.get("Etiquetas", "")) if pd.notna(row.get("Etiquetas")) else ""
    url_link = row.get("Url")

    # Añadir el enlace a la descripción si existe
    if pd.notna(url_link) and str(url_link).strip():
        descripcion_txt += f"<br><br><a href='{url_link}' target='_blank'>🔗 Más información</a>"

    # 4. Crear el panel H5P
    panel = {
        "panelTitle": original_name,
        "image": {
            "params": {
                "file": {"path": f"images/{image_filename}", "mime": "image/jpeg", "copyright": {"license": "U"}},
                "alt": f"Ilustración de {original_name}"
            },
            "library": "H5P.Image 1.1",
            "subContentId": str(uuid.uuid4()),
            "metadata": {"contentType": "Imagen", "license": "U", "title": original_name}
        },
        # MAPEO DE CONTENIDO: el orden aquí debe ser el mismo que en "properties"
        "entries": [
            titulo_txt,
            descripcion_txt,
            proyecto_txt,
            categoria_txt,
            etiquetas_txt
        ],
        "keywords": ""
    }
    h5p_template["infoWall"]["panels"].append(panel)

# --- Guardar resultados ---
with open('content.json', 'w', encoding='utf-8') as f:
    json.dump(h5p_template, f, indent=4, ensure_ascii=False)

zip_filename = 'h5p_actividades.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('content.json')
    for root, _, files in os.walk('images'):
        for file in files:
            zipf.write(os.path.join(root, file))

print(f"\n¡Proceso completado! Descarga el archivo '{zip_filename}'")

Procesando actividades desde el archivo ODS...
Procesando: Tablet Infantil y Primaria #1-99218
Error descargando imagen https://via.placeholder.com/400x267?text=Sin+Imagen: HTTPSConnectionPool(host='via.placeholder.com', port=443): Max retries exceeded with url: /400x267?text=Sin+Imagen (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x78fe41882630>: Failed to resolve 'via.placeholder.com' ([Errno -3] Temporary failure in name resolution)"))
Procesando: Tablet Infantil y Primaria #2-99219
Error descargando imagen https://via.placeholder.com/400x267?text=Sin+Imagen: HTTPSConnectionPool(host='via.placeholder.com', port=443): Max retries exceeded with url: /400x267?text=Sin+Imagen (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x78fe43fbf590>: Failed to resolve 'via.placeholder.com' ([Errno -3] Temporary failure in name resolution)"))
Procesando: Tablet Infantil y Primaria #3-99221
Error descargando imagen https://via.placeholde